# Лекция 13. Рекуррентные модели в глубоком обучении

**Тема:** `RNN`, `LSTM`, `GRU`, двунаправленные модели и `attention`  
**Формат:** теория + практические примеры в коде  
**Язык:** Python (`NumPy`, `PyTorch` при наличии)

---

Эта тетрадь рассчитана на занятие примерно **100-110 минут**.

Как пользоваться этой лекцией:

- Сначала читайте markdown-блок, потом запускайте следующий код-блок.
- В каждом разделе формулы сначала даны в математическом виде, а ниже расшифрованы «по-человечески».
- Если вы подзабыли математику: ориентируйтесь на смысл операций и формы тензоров, этого достаточно для практики.


## Для кого эта лекция

- Для студентов, которые уже знают базовые нейросети (MLP, backpropagation).
- Для тех, кто хочет понять, как работать с последовательностями: текст, временные ряды, события.

## Если математика подзабылась

В этой тетради мы используем минимальный набор идей:

- **Вектор**: просто список чисел (например, признаки токена).
- **Матрица**: таблица чисел, которая преобразует векторы.
- **Индекс `t`**: шаг времени в последовательности.
- **Нелинейность** (`tanh`, `sigmoid`): функция, которая помогает сети учить сложные зависимости.

Вам не нужно вручную выводить производные. Важно понимать:

1. что хранится в каждом объекте (`x_t`, `h_t`, `c_t`),
2. что модель делает на каждом шаге,
3. как интерпретировать выходы и метрики.

## Цели занятия

К концу лекции вы сможете:

1. Объяснить, зачем нужны рекуррентные модели и в чем их ограничения.
2. Различать `RNN`, `LSTM`, `GRU`, `Bidirectional RNN`.
3. Запускать и модифицировать минимальные примеры кода.
4. Выбрать разумную архитектуру под простую задачу последовательностей.


In [2]:
from __future__ import annotations

# Базовые библиотеки:
# math/random/numpy нужны почти в каждом ML-ноутбуке для чисел, генерации данных и векторных операций.
import math
import random
import numpy as np

# MPL_AVAILABLE: проверка, можно ли строить графики (matplotlib).
# Почему так: в некоторых средах (например, «голый» сервер) matplotlib может отсутствовать,
# и мы не хотим, чтобы ноутбук падал из-за визуализации.
MPL_AVAILABLE = True
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    MPL_AVAILABLE = False
    print(f"matplotlib недоступен: {exc}")

# SEED (зерно генератора случайных чисел) фиксируем для воспроизводимости.
# Почему так: результаты обучения/генерации будут повторяться от запуска к запуску.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# TORCH_AVAILABLE: проверка, доступен ли PyTorch.
# Почему так: часть практики использует нейросети из torch, но теоретические части
# должны работать даже без него.
TORCH_AVAILABLE = True
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(SEED)
except Exception as exc:
    TORCH_AVAILABLE = False
    print(f"PyTorch недоступен: {exc}")

print(f"matplotlib available: {MPL_AVAILABLE}")
print(f"PyTorch available: {TORCH_AVAILABLE}")


matplotlib available: True
PyTorch available: True


## 1. Зачем вообще рекуррентные модели

Во многих задачах важен **порядок** и **контекст во времени**:

- текст (следующее слово зависит от предыдущих),
- временной ряд (текущее значение зависит от прошлых),
- последовательности событий (логи, клики, транзакции),
- речь и аудио.

Почему обычный MLP неудобен для этого:

- ему нужен вход фиксированной длины,
- он не хранит «память» о прошлых шагах,
- при длинных последовательностях теряется структура «что было раньше».

Идея RNN: на каждом шаге мы обновляем скрытое состояние `h_t`, которое хранит сжатую память о прошлом.

Простая аналогия:

- `x_t` — текущее слово в предложении,
- `h_{t-1}` — то, что модель «помнит» до этого слова,
- `h_t` — обновленная память после чтения текущего слова.

Так сеть читает последовательность слева направо и постепенно накапливает контекст.


![RNN intuition](images/rnn_effectiveness.jpeg)

Как читать картинку:

- одинаковые блоки повторяются по времени — это одна и та же ячейка RNN с общими весами,
- стрелка вперед по времени показывает передачу состояния,
- выход на текущем шаге зависит от текущего входа и прошлого состояния.

*Интуиция: модель обрабатывает токены последовательно и переносит состояние между шагами.*


## 2. Базовая RNN

На шаге $t$:

$$
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)
$$

$$
y_t = W_{hy}h_t + b_y
$$

### Расшифровка формул на простом языке

- Сначала берем текущий вход $x_t$ и прошлую память $h_{t-1}$.
- Линейно смешиваем их через матрицы весов $W_{xh}$ и $W_{hh}$.
- Добавляем смещение $b_h$.
- Пропускаем через $\tanh$ и получаем новую память $h_t$.
- Из $h_t$ строим выход $y_t$.

### Что означают символы

- $t$ — номер шага времени (например, номер токена в предложении).
- Нижний индекс: `t-1` = «предыдущий шаг».
- $W$ — матрица обучаемых весов.
- $b$ — bias (смещение), тоже обучаемый параметр.
- $\tanh(\,)$ — нелинейность, сжимает значения примерно в диапазон `[-1, 1]`.

### Зачем именно так

- Одни и те же матрицы используются на каждом шаге: модель может работать с последовательностями разной длины.
- Состояние $h_t$ играет роль «краткой памяти».
- Это минимальная архитектура, от которой логично перейти к LSTM/GRU.


In [ ]:
# Forward-pass (прямой проход) простой RNN на игрушечной последовательности (NumPy)

# vocab (словарь): уникальные токены, с которыми работаем.
vocab = ["я", "люблю", "машинное", "обучение"]
# token_to_id: преобразование токена в индекс, чтобы перейти к числам.
token_to_id = {t: i for i, t in enumerate(vocab)}

# sequence: конкретная входная последовательность.
sequence = ["я", "люблю", "машинное", "обучение"]

# T = число шагов по времени (time steps)
# V = размер словаря (vocabulary size)
# H = размер скрытого состояния (hidden size, "объем памяти" RNN)
T = len(sequence)
V = len(vocab)
H = 6

# One-hot кодирование: каждое слово = вектор длины V,
# где 1 стоит на позиции слова, остальные 0.
# Почему так: это простейший способ подать категориальные токены в линейную алгебру.
X = np.zeros((T, V))
for t, token in enumerate(sequence):
    X[t, token_to_id[token]] = 1.0

# Параметры RNN (случайная инициализация для демонстрации):
# Wxh: вход -> скрытое состояние
# Whh: прошлое скрытое -> новое скрытое
# Why: скрытое состояние -> выход
# b*: смещения (bias)
Wxh = np.random.randn(H, V) * 0.3
Whh = np.random.randn(H, H) * 0.3
Why = np.random.randn(V, H) * 0.3
bh = np.zeros(H)
by = np.zeros(V)

# h: текущее скрытое состояние, сначала нулевое (нет контекста до первого токена).
h = np.zeros(H)
hidden_states = []  # будем сохранять h_t для каждого шага t
logits = []         # logits = «сырые» оценки до softmax

for t in range(T):
    x_t = X[t]  # one-hot вектор текущего токена

    # Формула RNN-ячейки: h_t = tanh(Wxh*x_t + Whh*h_{t-1} + b)
    # Почему tanh: ограничивает диапазон активаций и вводит нелинейность.
    h = np.tanh(Wxh @ x_t + Whh @ h + bh)

    # y_t (logits) можно было бы подать в softmax для вероятностей классов.
    y_t = Why @ h + by

    hidden_states.append(h.copy())
    logits.append(y_t.copy())

hidden_states = np.array(hidden_states)
print('Форма матрицы скрытых состояний:', hidden_states.shape)

if MPL_AVAILABLE:
    plt.figure(figsize=(9, 3))
    plt.imshow(hidden_states.T, aspect='auto', cmap='coolwarm')
    plt.colorbar(label='activation')
    plt.yticks(range(H), [f'h{i}' for i in range(H)])
    plt.xticks(range(T), sequence)
    plt.title('Скрытые состояния RNN по времени')
    plt.xlabel('Шаг времени (токен)')
    plt.ylabel('Нейрон скрытого состояния')
    plt.show()
else:
    print('График пропущен: matplotlib недоступен.')


Что важно заметить после примера с `hidden_states`:

- Состояние `h_t` меняется после каждого токена.
- Вектор `h_t` должен сжать полезный контекст о предыдущих шагах.
- Чем длиннее зависимость, тем тяжелее базовой RNN удерживать информацию без потерь.

Как читать матрицу скрытых состояний:

- `hidden_states` имеет форму `(T, H)`.
- `T` — число шагов времени, `H` — число скрытых нейронов.
- Элемент `hidden_states[t, j]` — активация нейрона `j` на шаге `t`.

Если смотреть heatmap `hidden_states.T`:

- ось X — время (токены),
- ось Y — номер нейрона,
- цвет — сила и знак активации.

Практический вывод: смотрите не «один цвет», а динамику строк/столбцов — так видно, какие нейроны реагируют на конкретные позиции.


## 3. Обучение через время (BPTT) и проблема градиентов

`BPTT` (Backpropagation Through Time) — это обычный backpropagation, но примененный к RNN, развернутой по шагам времени.

Идея простыми словами:

1. Мы считаем ошибки на выходах.
2. «Прокатываем» их назад через все шаги `t, t-1, t-2, ...`.
3. Обновляем общие веса `W`, которые использовались на каждом шаге.

Почему возникают проблемы:

- при длинной цепочке градиент много раз умножается на матрицы производных,
- если множители чаще меньше 1, получаем **затухание** (vanishing gradient),
- если множители чаще больше 1, получаем **взрыв** (exploding gradient).

Расшифровка терминов:

- **Градиент** — «насколько и в какую сторону менять веса, чтобы уменьшить ошибку».
- **Якобиан** — матрица частных производных для векторной функции (в практике важно как «локальный коэффициент изменения»).

Практический смысл для студента:

- базовая RNN обычно плохо держит очень дальние зависимости,
- поэтому и появились `LSTM`/`GRU`, где поток информации контролируется воротами.


In [ ]:
# Визуализация затухания/взрыва градиента как повторного умножения коэффициента.
# Градиент = насколько сильно ошибка влияет на параметры.
# При длинных цепочках (BPTT) такие множители перемножаются много раз.

steps = np.arange(1, 51)

# vanishing gradient (затухающие градиенты): множитель < 1
vanishing = 0.6 ** steps
# условно стабильный случай
stable = 1.0 ** steps
# exploding gradient (взрывающиеся градиенты): множитель > 1
exploding = 1.2 ** steps

if MPL_AVAILABLE:
    plt.figure(figsize=(8, 4))
    plt.plot(steps, vanishing, label='0.6^t (затухание)')
    plt.plot(steps, stable, label='1.0^t (стабильно)')
    plt.plot(steps, exploding, label='1.2^t (взрыв)')
    plt.yscale('log')
    plt.xlabel('t (длина зависимости)')
    plt.ylabel('масштаб градиента (log)')
    plt.title('Почему у длинных последовательностей возникают проблемы')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('matplotlib недоступен; первые значения:')
    print('vanishing:', np.round(vanishing[:8], 6))
    print('exploding:', np.round(exploding[:8], 6))


## 4. LSTM: управляемая память

`LSTM` (Long Short-Term Memory) — это RNN с более устойчивой памятью.

Главная идея:

- есть обычное скрытое состояние `h_t` (что отдать наружу сейчас),
- и отдельная «долгая» память `c_t` (что хранить дольше).

Поток информации регулируют ворота (gates):

- `f_t` (forget gate): какую часть старой памяти оставить,
- `i_t` (input gate): сколько новой информации записать,
- `o_t` (output gate): какую часть памяти показать во внешнем состоянии `h_t`.

Почему это работает лучше RNN:

- модель умеет «пропускать важное дальше» и «гасить шум»,
- градиенту проще проходить через длинные цепочки,
- на практике лучше для длинных текстов/сигналов.


![LSTM cell](images/lstm_cell.svg)

Классические формулы LSTM:

$$
f_t = \sigma(W_f [x_t, h_{t-1}] + b_f), \quad
i_t = \sigma(W_i [x_t, h_{t-1}] + b_i)
$$

$$
\tilde{c}_t = \tanh(W_c [x_t, h_{t-1}] + b_c), \quad
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

$$
o_t = \sigma(W_o [x_t, h_{t-1}] + b_o), \quad
h_t = o_t \odot \tanh(c_t)
$$

### Расшифровка спецсимволов

- $\sigma$ (сигма) — `sigmoid`, значения от 0 до 1.
  Практически: «клапан», который говорит, сколько информации пропустить.
- $\tanh$ — нелинейность с диапазоном примерно `[-1, 1]`.
- $\tilde{c}_t$ (читается «си с тильдой») — кандидат новой памяти.
- $\odot$ — поэлементное умножение (Hadamard product): умножаем соответствующие элементы векторов.
- `[x_t, h_{t-1}]` — конкатенация (склейка) двух векторов в один.

### Пошагово без математики

1. Считаем `forget gate`: сколько старой памяти оставить.
2. Считаем `input gate` и кандидат: сколько новой информации добавить.
3. Обновляем память `c_t` как смесь старой и новой.
4. Через `output gate` формируем `h_t`, которое пойдет дальше в сеть.

Именно этот механизм делает LSTM устойчивее на длинных последовательностях.


In [ ]:
# Один шаг LSTM вручную (NumPy)
# LSTM = Long Short-Term Memory (долгая краткосрочная память).
# Идея: добавить «ворота» (gates), которые управляют, что помнить/забывать.

def sigmoid(x):
    # sigmoid сжимает значения в [0, 1], поэтому удобно интерпретировать как «долю пропуска».
    return 1.0 / (1.0 + np.exp(-x))

# input_size: размер входного вектора x_t
# hidden_size: размер скрытого состояния h_t и состояния памяти c_t
input_size = 4
hidden_size = 3

x_t = np.random.randn(input_size)     # вход на текущем шаге времени
h_prev = np.zeros(hidden_size)        # прошлое скрытое состояние h_{t-1}
c_prev = np.zeros(hidden_size)        # прошлое состояние ячейки c_{t-1} (долгая память)

# Для простоты объединим все 4 набора весов LSTM в одну большую матрицу W:
# [i, f, g, o] по строкам.
# Почему так: это стандартная векторизация, быстрее и компактнее.
W = np.random.randn(4 * hidden_size, input_size + hidden_size) * 0.2
b = np.zeros(4 * hidden_size)

# concat = [x_t ; h_prev] — объединяем текущий вход и прошлую память.
concat = np.concatenate([x_t, h_prev])
gates = W @ concat + b

# Разбиваем на 4 блока:
# i = input gate (что записать)
# f = forget gate (что забыть)
# g = candidate (кандидат новой информации)
# o = output gate (что отдать во внешнее h_t)
i, f, g, o = np.split(gates, 4)
i = sigmoid(i)
f = sigmoid(f)
g = np.tanh(g)
o = sigmoid(o)

# Обновление памяти c_t и скрытого состояния h_t
c_t = f * c_prev + i * g
h_t = o * np.tanh(c_t)

print('i (input gate):', np.round(i, 3))
print('f (forget gate):', np.round(f, 3))
print('o (output gate):', np.round(o, 3))
print('h_t:', np.round(h_t, 3))


![Multi-layer LSTM](images/multi_layer_lstm.jpeg)

Что происходит в многослойной LSTM:

- первый слой учит базовые паттерны (локальные зависимости),
- следующий слой строит более абстрактные признаки,
- глубина помогает качеству, но повышает риск переобучения и время обучения.

Почему используют дополнительные техники:

- `dropout` между слоями — снижает переобучение,
- `bidirectional` — учитывает и левый, и правый контекст,
- `gradient clipping` — стабилизирует обучение.

Если вы забыли математику: думайте о слоях как о «последовательной очистке сигнала», где каждый слой уточняет представление.


## 5. GRU: более компактная альтернатива

`GRU` (Gated Recurrent Unit) — упрощенная версия LSTM.

Интуитивно:

- в GRU меньше ворот и параметров,
- модель быстрее обучается,
- качество часто близко к LSTM на реальных задачах.

Типичные формулы GRU:

$$
z_t = \sigma(W_z [x_t, h_{t-1}] + b_z), \quad
r_t = \sigma(W_r [x_t, h_{t-1}] + b_r)
$$

$$
\tilde{h}_t = \tanh(W_h [x_t, r_t \odot h_{t-1}] + b_h)
$$

$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

Расшифровка:

- `z_t` (update gate) — насколько обновить состояние,
- `r_t` (reset gate) — насколько учитывать прошлое при создании кандидата,
- `\tilde{h}_t` — кандидат нового состояния,
- финальный `h_t` — смесь старой памяти и новой.

Почему именно так: GRU сохраняет идею «управляемой памяти», но делает её компактнее.


In [ ]:
# Сравним количество параметров у RNN / GRU / LSTM
# Почему это важно: больше параметров = потенциально выше качество,
# но медленнее обучение и выше риск переобучения.

def params_rnn(input_size: int, hidden_size: int) -> int:
    # Один набор весов + bias для перехода по времени.
    return hidden_size * input_size + hidden_size * hidden_size + hidden_size


def params_gru(input_size: int, hidden_size: int) -> int:
    # GRU имеет 3 «гейта/блока» параметров (update/reset/new).
    return 3 * (hidden_size * input_size + hidden_size * hidden_size + hidden_size)


def params_lstm(input_size: int, hidden_size: int) -> int:
    # LSTM имеет 4 блока параметров (i, f, g, o), поэтому обычно тяжелее.
    return 4 * (hidden_size * input_size + hidden_size * hidden_size + hidden_size)


input_size, hidden_size = 64, 128
cells_names = ['RNN', 'GRU', 'LSTM']
counts = [
    params_rnn(input_size, hidden_size),
    params_gru(input_size, hidden_size),
    params_lstm(input_size, hidden_size),
]

for name, count in zip(cells_names, counts):
    print(f'{name:4s}: {count:,}')

if MPL_AVAILABLE:
    plt.figure(figsize=(7, 4))
    plt.bar(cells_names, counts, color=['#4C78A8', '#F58518', '#54A24B'])
    for i, v in enumerate(counts):
        plt.text(i, v, f'{v:,}', ha='center', va='bottom')
    plt.title(f'Количество параметров (input={input_size}, hidden={hidden_size})')
    plt.ylabel('Параметры')
    plt.show()
else:
    print('Гистограмма пропущена: matplotlib недоступен.')


## 6. Практика на реальном датасете: сравнение RNN / GRU / LSTM (IMDb)

Теперь уходим от синтетики и делаем «живой» эксперимент на **реальных текстах**.

Датасет: `aclImdb` (50k отзывов на фильмы, бинарная тональность).

Что делаем в этом разделе:

1. Загружаем локальные тексты из `data/aclImdb/train|test/(pos|neg)`.
2. Строим словарь и переводим тексты в индексы токенов.
3. Обучаем 3 модели (`RNN`, `GRU`, `LSTM`) в одинаковых условиях.
4. Сравниваем `accuracy` на тесте и динамику обучения.

Почему это «более живо»:

- тексты реальные и шумные,
- длины и стиль отзывов сильно различаются,
- видно реальные ограничения моделей и гиперпараметров.

Важно: чтобы пример работал быстро на ноутбуке, в коде берется **подмножество** IMDb.


In [ ]:
TRAIN_RESULTS = None
TRAINED_MODELS = {}
REAL_TEXT_CTX = {}

if not TORCH_AVAILABLE:
    print('Пропуск: PyTorch не установлен, этот блок не выполняется.')
else:
    import re
    from collections import Counter
    from pathlib import Path
    from torch.nn.utils.rnn import pack_padded_sequence

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device:', device)

    def find_imdb_root() -> Path:
        """Ищем imdb-датасет в нескольких типичных местах."""
        candidates = [
            Path('data/aclImdb'),
            Path('../../data/aclImdb'),
            Path('/Users/hiber/repositories/courses/ml/data/aclImdb'),
        ]
        for c in candidates:
            if (c / 'train' / 'pos').exists() and (c / 'test' / 'neg').exists():
                return c.resolve()
        raise FileNotFoundError('Не найден aclImdb. Проверьте путь data/aclImdb')


    imdb_root = find_imdb_root()
    print('IMDb root:', imdb_root)

    MAX_TRAIN_PER_CLASS = 2500  # чтобы запускался разумно быстро
    MAX_TEST_PER_CLASS = 1000
    MAX_LEN = 180               # длина последовательности в токенах
    MAX_VOCAB = 20000           # размер словаря
    MIN_FREQ = 2

    PAD_IDX = 0
    UNK_IDX = 1

    def clean_text(text: str) -> str:
        return text.replace('<br />', ' ').replace('\n', ' ').strip()


    def tokenize(text: str):
        """Простая токенизация: слова + знаки пунктуации."""
        return re.findall(r"[a-z']+|[!?.,;:]", text.lower())


    def load_imdb_split(root: Path, split: str, max_per_class: int):
        texts, labels = [], []
        # 0 = neg, 1 = pos
        for label_name, label_id in [('neg', 0), ('pos', 1)]:
            files = sorted((root / split / label_name).glob('*.txt'))[:max_per_class]
            for fp in files:
                txt = fp.read_text(encoding='utf-8', errors='ignore')
                texts.append(clean_text(txt))
                labels.append(label_id)
        return texts, labels


    train_texts, train_labels = load_imdb_split(imdb_root, 'train', MAX_TRAIN_PER_CLASS)
    test_texts, test_labels = load_imdb_split(imdb_root, 'test', MAX_TEST_PER_CLASS)

    print(f'Train size: {len(train_texts)} | Test size: {len(test_texts)}')
    print('Пример отзыва (train):', train_texts[0][:220], '...')

    def build_vocab(texts, max_vocab: int = 20000, min_freq: int = 2):
        counter = Counter()
        for text in texts:
            counter.update(tokenize(text))

        itos = ['<pad>', '<unk>']
        for tok, freq in counter.most_common():
            if freq < min_freq:
                break
            if len(itos) >= max_vocab:
                break
            itos.append(tok)

        stoi = {tok: i for i, tok in enumerate(itos)}
        return stoi, itos


    stoi, itos = build_vocab(train_texts, max_vocab=MAX_VOCAB, min_freq=MIN_FREQ)
    print('Vocab size:', len(itos))

    def encode_text(text: str, stoi_map, max_len: int):
        toks = tokenize(text)[:max_len]
        ids = [stoi_map.get(t, UNK_IDX) for t in toks]
        length = len(ids)
        if length < max_len:
            ids.extend([PAD_IDX] * (max_len - length))
        return ids, length


    def make_tensor_dataset(texts, labels, stoi_map, max_len: int):
        all_ids, all_len = [], []
        for txt in texts:
            ids, ln = encode_text(txt, stoi_map, max_len)
            all_ids.append(ids)
            all_len.append(ln)

        X = torch.tensor(all_ids, dtype=torch.long)
        L = torch.tensor(all_len, dtype=torch.long)
        y = torch.tensor(labels, dtype=torch.long)
        return TensorDataset(X, L, y)


    train_ds = make_tensor_dataset(train_texts, train_labels, stoi, MAX_LEN)
    test_ds = make_tensor_dataset(test_texts, test_labels, stoi, MAX_LEN)

    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=128, shuffle=False)


    class TextRNNClassifier(nn.Module):
        """Классификатор тональности на RNN/GRU/LSTM поверх эмбеддингов токенов."""

        def __init__(self, vocab_size: int, cell_type: str = 'GRU', embed_dim: int = 96, hidden_size: int = 96):
            super().__init__()
            self.cell_type = cell_type.upper()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

            if self.cell_type == 'RNN':
                self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
            elif self.cell_type == 'GRU':
                self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
            elif self.cell_type == 'LSTM':
                self.rnn = nn.LSTM(embed_dim, hidden_size, batch_first=True)
            else:
                raise ValueError(f'Unknown cell type: {cell_type}')

            self.dropout = nn.Dropout(0.2)
            self.head = nn.Linear(hidden_size, 2)

        def forward(self, x, lengths):
            emb = self.embedding(x)
            packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, hidden = self.rnn(packed)

            if self.cell_type == 'LSTM':
                h_last = hidden[0][-1]
            else:
                h_last = hidden[-1]

            logits = self.head(self.dropout(h_last))
            return logits


    def evaluate(model: nn.Module, loader: DataLoader):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, lb, yb in loader:
                xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
                pred = model(xb, lb).argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        return correct / max(total, 1)


    def train_one_model(cell_type: str, epochs: int = 2, lr: float = 2e-3):
        model = TextRNNClassifier(vocab_size=len(itos), cell_type=cell_type).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        history = []

        for epoch in range(1, epochs + 1):
            model.train()
            total_loss = 0.0
            total_items = 0

            for xb, lb, yb in train_dl:
                xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
                optimizer.zero_grad()
                logits = model(xb, lb)
                loss = criterion(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                batch_size = yb.size(0)
                total_loss += loss.item() * batch_size
                total_items += batch_size

            train_loss = total_loss / max(total_items, 1)
            test_acc = evaluate(model, test_dl)
            history.append((train_loss, test_acc))
            print(f"{cell_type:4s} | epoch {epoch:02d} | loss={train_loss:.4f} | test_acc={test_acc:.3f}")

        return model, history


    REAL_TEXT_CTX = {
        'imdb_root': str(imdb_root),
        'stoi': stoi,
        'itos': itos,
        'max_len': MAX_LEN,
        'pad_idx': PAD_IDX,
        'unk_idx': UNK_IDX,
        'train_texts': train_texts,
        'train_labels': train_labels,
        'test_texts': test_texts,
        'test_labels': test_labels,
        'device': str(device),
    }

    print('Подготовка завершена: real IMDb -> DataLoader -> модели готовы.')


In [ ]:
if not TORCH_AVAILABLE:
    print('Пропуск: PyTorch не установлен.')
elif 'train_one_model' not in globals():
    print('Сначала выполните предыдущую ячейку с подготовкой IMDb.')
else:
    results = {}
    models = {}

    for cell_type in ['RNN', 'GRU', 'LSTM']:
        model, history = train_one_model(cell_type, epochs=2, lr=2e-3)
        models[cell_type] = model
        results[cell_type] = history

    TRAINED_MODELS = models
    TRAIN_RESULTS = results

    if MPL_AVAILABLE:
        plt.figure(figsize=(8, 4))
        for cell_type, history in results.items():
            acc = [x[1] for x in history]
            plt.plot(range(1, len(acc) + 1), acc, marker='o', label=cell_type)

        plt.title('IMDb subset: сравнение test accuracy')
        plt.xlabel('Эпоха')
        plt.ylabel('Accuracy')
        plt.ylim(0.45, 1.01)
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()

    print('\nФинальные качества на IMDb subset:')
    for cell_type, history in results.items():
        print(f"{cell_type:4s}: {history[-1][1]:.3f}")

    best_type = max(results, key=lambda k: results[k][-1][1])
    REAL_TEXT_CTX['best_model_type'] = best_type
    print(f"\nЛучший тип на текущем прогоне: {best_type}")


## 7. Bidirectional RNN / LSTM

Идея: пройти последовательность в двух направлениях:

- слева направо,
- справа налево.

После этого два представления объединяются.

Плюс подхода:

- каждое положение «видит» и левый, и правый контекст,
- особенно полезно для задач теггинга текста, NER, классификации фраз,
- часто повышает качество там, где доступна вся последовательность заранее.

Важно понимать ограничение:

- для онлайн-режима (стриминг, автодополнение «на лету») BiRNN не всегда применима,
  потому что ей нужен будущий контекст.


In [ ]:
if not TORCH_AVAILABLE:
    print('Пропуск: PyTorch не установлен.')
else:
    # Bidirectional LSTM = LSTM в двух направлениях (слева-направо и справа-налево).
    # Почему это полезно: представление токена учитывает и левый, и правый контекст.
    bi_lstm = nn.LSTM(
        input_size=8,       # число признаков на один шаг времени
        hidden_size=16,     # размер скрытого состояния одного направления
        num_layers=1,       # число слоев LSTM
        batch_first=True,   # формат входа: (batch, seq_len, features)
        bidirectional=True, # ключевой флаг: включаем два направления
    )

    x = torch.randn(4, 10, 8)  # batch=4, seq_len=10, features=8
    out, (h, c) = bi_lstm(x)

    # out: выход на каждом шаге -> (batch, seq_len, 2*hidden)
    # 2*hidden, потому что склеиваются forward и backward направления.
    print('out.shape:', tuple(out.shape))

    # h: последние скрытые состояния по направлениям -> (2, batch, hidden)
    print('h.shape:', tuple(h.shape))

    # Берем 2 последних состояния (forward+backward),
    # переносим batch вперед и склеиваем в один вектор признаков на объект.
    h_last = h[-2:].transpose(0, 1).reshape(4, -1)
    print('Склеенное представление двух направлений:', tuple(h_last.shape))


## 8. Seq2Seq и attention: где это связано с рекуррентными моделями

Классический `seq2seq` (до эпохи трансформеров):

1. Encoder-RNN читает входную последовательность.
2. Decoder-RNN генерирует выходную последовательность.
3. Attention помогает декодеру выбирать, на какие части входа смотреть на каждом шаге.

Без attention у декодера часто «узкое горлышко»: весь смысл входа надо упаковать в один вектор.
С attention модель получает динамический доступ к разным позициям входа.

Мини-формула attention:

$$
	ext{score}_i = rac{q^T k_i}{\sqrt{d}}, \quad
lpha_i = 	ext{softmax}(	ext{score}_i), \quad
c = \sum_i lpha_i v_i
$$

Расшифровка символов:

- $q$ (`query`) — «что ищем сейчас».
- $k_i$ (`key`) — «описание» позиции `i` во входе.
- $v_i$ (`value`) — информация, которую берем из позиции `i`.
- $q^T k_i$ — скалярное произведение (мера похожести).
- $\sqrt{d}$ — нормализация масштаба.
- $lpha_i$ — вес внимания к позиции `i`, после `softmax` все веса суммируются в 1.
- $c$ — итоговый контекст как взвешенная сумма.

Ниже делаем живой пример: мини-модель **GRU + attention** на IMDb и визуализация,
на какие слова конкретного отзыва модель смотрит сильнее.


![Encoder-Decoder Attention](images/encoder_decoder_attention.png)

![RNN text generation](images/rnn_generate.png)

Как читать эти схемы:

- на верхней: декодер получает не один фиксированный контекст, а «переоценивает внимание» на каждом шаге,
- на нижней: генерация текста идет токен за токеном, и качество зависит от того, насколько хорошо модель держит контекст.


In [ ]:
# Attention на реальном тексте IMDb: обучаем мини-модель и смотрим веса слов

if not TORCH_AVAILABLE:
    print('Пропуск: PyTorch не установлен.')
elif 'train_dl' not in globals() or 'tokenize' not in globals():
    print('Сначала выполните ячейку подготовки IMDb (раздел 6).')
else:
    from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class AttentionSentimentModel(nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int = 96, hidden_size: int = 96):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
            self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True, bidirectional=True)

            # Additive attention: score_t = v^T tanh(W h_t)
            self.attn_proj = nn.Linear(hidden_size * 2, hidden_size * 2)
            self.attn_vec = nn.Linear(hidden_size * 2, 1, bias=False)

            self.dropout = nn.Dropout(0.2)
            self.head = nn.Linear(hidden_size * 2, 2)

        def forward(self, x, lengths):
            emb = self.embedding(x)
            packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.gru(packed)
            out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))

            scores = self.attn_vec(torch.tanh(self.attn_proj(out))).squeeze(-1)

            # Маска паддинга: на позициях beyond length веса внимания должны быть 0.
            max_t = x.size(1)
            mask = torch.arange(max_t, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)
            scores = scores.masked_fill(mask, -1e9)

            weights = torch.softmax(scores, dim=1)
            context = torch.sum(out * weights.unsqueeze(-1), dim=1)
            logits = self.head(self.dropout(context))
            return logits, weights


    def evaluate_attention_model(model, loader):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, lb, yb in loader:
                xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
                logits, _ = model(xb, lb)
                pred = logits.argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        return correct / max(total, 1)


    attn_model = AttentionSentimentModel(vocab_size=len(itos)).to(device)
    optimizer = torch.optim.Adam(attn_model.parameters(), lr=2e-3)
    criterion = nn.CrossEntropyLoss()

    # Быстрое демо-обучение: 1 эпоха (можно увеличить для лучшего качества).
    attn_model.train()
    running_loss, total_items = 0.0, 0
    for xb, lb, yb in train_dl:
        xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits, _ = attn_model(xb, lb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(attn_model.parameters(), 1.0)
        optimizer.step()

        bs = yb.size(0)
        running_loss += loss.item() * bs
        total_items += bs

    attn_test_acc = evaluate_attention_model(attn_model, test_dl)
    print(f'Attention model | loss={running_loss / max(total_items,1):.4f} | test_acc={attn_test_acc:.3f}')

    # Покажем веса внимания на одном реальном отзыве.
    sample_idx = 0
    sample_text = REAL_TEXT_CTX['test_texts'][sample_idx]
    sample_label = REAL_TEXT_CTX['test_labels'][sample_idx]

    ids, ln = encode_text(sample_text, stoi, REAL_TEXT_CTX['max_len'])
    x = torch.tensor([ids], dtype=torch.long, device=device)
    l = torch.tensor([ln], dtype=torch.long, device=device)

    attn_model.eval()
    with torch.no_grad():
        logits, weights = attn_model(x, l)
        prob = torch.softmax(logits, dim=1)[0, 1].item()

    pred_label = int(prob >= 0.5)
    print('\nПример реального отзыва IMDb (обрезан):')
    print(sample_text[:320].replace('\n', ' '), '...')
    print(f'Истинная метка: {sample_label} | Предсказание: {pred_label} | P(pos)={prob:.3f}')

    toks = tokenize(sample_text)[:ln]
    w = weights[0, :ln].detach().cpu().numpy()

    # Топ-10 токенов по вниманию
    top_idx = w.argsort()[-10:][::-1]
    print('\nТоп токенов по attention:')
    for rank, idx in enumerate(top_idx, start=1):
        print(f'{rank:2d}. {toks[idx]:<15s} weight={w[idx]:.4f}')

    if MPL_AVAILABLE:
        show_n = min(20, ln)
        plt.figure(figsize=(12, 3))
        plt.bar(range(show_n), w[:show_n])
        plt.xticks(range(show_n), toks[:show_n], rotation=60, ha='right')
        plt.title('Attention-веса (первые токены отзыва)')
        plt.tight_layout()
        plt.show()


## 9. Практические советы и частые ошибки

### Что обычно работает

- Начинайте с `GRU` как сильного и компактного базового варианта.
- Для очень длинных зависимостей чаще берут `LSTM` или attention-подходы.
- Используйте `gradient clipping` (особенно для RNN/LSTM).
- Нормализуйте входы и аккуратно выбирайте `sequence length`.
- Контролируйте баланс классов: плохой баланс легко ломает метрики.

### Типичные проблемы

- Перепутан порядок тензоров (`batch_first=True/False`).
- Слишком длинные последовательности без обрезки или bucketing.
- Плохой batching (слишком маленький batch => шумная оптимизация).
- Отсутствие валидационного контроля и ранней остановки.
- Сравнение моделей при разных гиперпараметрах (некорректный эксперимент).

### Мини-памятка «если забыли математику»

- Смотрите на формы тензоров (`shape`) чаще, чем на формулы.
- Держите в голове простой цикл: вход -> скрытое состояние -> выход -> ошибка -> обновление весов.
- Проверяйте интерпретацию метрик: `loss` и `accuracy` должны двигаться согласованно.


## 10. Упражнения на занятии (на реальном IMDb)

1. Увеличьте `MAX_TRAIN_PER_CLASS` в 2 раза. Как меняется `accuracy` и время эпохи?
2. Измените `MAX_LEN` (например, 80, 180, 300). Где баланс качества/скорости лучший?
3. Сравните `RNN`, `GRU`, `LSTM` при одинаковом `embed_dim` и `hidden_size`.
4. Для attention-модели покажите топ-10 слов для позитивного и негативного отзыва.
5. Проверьте, какие ошибки модель делает чаще: короткие или длинные отзывы.

Что фиксировать в отчете:

- точные настройки (`MAX_LEN`, `MAX_VOCAB`, `epochs`, `hidden_size`),
- финальные метрики,
- 2-3 примера правильных и ошибочных предсказаний,
- вывод, какая архитектура практичнее в вашем ограничении по времени/ресурсам.


In [ ]:
# Шаблон для самостоятельного мини-эксперимента на IMDb

if not TORCH_AVAILABLE:
    print('PyTorch недоступен: выполните упражнение в окружении с torch.')
elif 'train_one_model' not in globals():
    print('Сначала выполните ячейку подготовки IMDb в разделе 6.')
else:
    # Гиперпараметры эксперимента
    epochs = 2
    lr = 2e-3
    cell_type = 'GRU'   # попробуйте: 'RNN', 'GRU', 'LSTM'

    print('Ваш эксперимент:')
    print(f'cell_type={cell_type}, epochs={epochs}, lr={lr}')

    model, history = train_one_model(cell_type=cell_type, epochs=epochs, lr=lr)
    print('История (loss, test_acc):', history)

    # TODO 1: попробуйте другой cell_type и сравните test_acc
    # TODO 2: измените MAX_LEN / MAX_VOCAB в ячейке подготовки и переобучите
    # TODO 3: сформулируйте вывод, что важнее для качества в вашем случае


## 11. Итоги

Сегодня разобрали:

- как работает базовая `RNN`,
- почему возникают проблемы градиента,
- как `LSTM` и `GRU` улучшают память,
- зачем нужен `Bidirectional` режим,
- как attention дополняет рекуррентные подходы.

Короткая «карта выбора» модели:

- `RNN` — учебный базис и простые короткие последовательности.
- `GRU` — хороший старт в большинстве практических задач.
- `LSTM` — когда важны длинные зависимости и устойчивость памяти.
- `BiLSTM` — когда доступен полный контекст слева и справа.
- `Attention/Seq2Seq` — когда нужно гибко фокусироваться на частях входа.

### Домашняя работа (вариант)

1. Подготовить задачу классификации коротких текстов на русском.
2. Обучить 3 модели: `RNN`, `GRU`, `LSTM`.
3. Сравнить по `accuracy`, времени эпохи и числу параметров.
4. Кратко описать, где каждая архитектура проигрывает/выигрывает.


## Дополнительные материалы

- `word2vec.py`, `rnn_simple.py`, `lstm.py`, `bidirectional_lstm.py` в этой же папке.
- Следующая логичная тема курса: `Attention` и `Transformer` (у вас уже есть отдельные лекции).

Рекомендация по повторению для тех, кто «выпал» из математики:

1. Снова пройти разделы 2, 4 и 8 этой тетради.
2. Запустить код и понаблюдать за `shape` и значениями после каждого шага.
3. Затем вернуться к формулам — обычно после практики они читаются проще.
